# Train Smoke Test
Run a short training loop and print losses.


In [1]:
import torch
from train import train, get_device
from prior_data import PriorGeneratorConfig
from model import CustomNanoTabPFNModel
from eval_pu_osls import evaluate_pu_osls, EvalConfig, print_results


# Training/eval knobs (tuned for a single 4090, <12h target)
train_steps = 200
batch_size = 32
lr = 5e-4
embedding_size = 64
num_attention_heads = 4
mlp_hidden_size = 128
num_layers = 3

eval_interval = 0
eval_cfg = EvalConfig(
    n_tasks=200,
    batch_size=16,
    seed=999,
    outlier_score='p_unseen',
)
cfg = PriorGeneratorConfig(
    max_classes=3,
    min_features=3,
    max_features=4,
    min_rows=100,
    max_rows=200,
    min_train_fraction=0.4,
    max_train_fraction=0.6,
    remove_poisson_lambda=1.0,
    seed=0,
    label_noise=0.1,
)

device = get_device()
unseen_label = cfg.max_classes
num_outputs = cfg.max_classes + 1

model = CustomNanoTabPFNModel(
    embedding_size=embedding_size,
    num_attention_heads=num_attention_heads,
    mlp_hidden_size=mlp_hidden_size,
    num_layers=num_layers,
    num_outputs=num_outputs,
    unseen_label=unseen_label,
)


In [2]:
model, losses = train(
    model,
    cfg,
    batch_size=batch_size,
    lr=lr,
    device=device,
    num_steps=train_steps,
    unseen_label=unseen_label,
    eval_interval=eval_interval
)

print('losses:', losses)
print('loss delta:', losses[0], '->', losses[-1])


eval_cfg = EvalConfig(
    n_tasks=20,
    batch_size=batch_size,
    seed=999,
    outlier_score='p_unseen',
)

print('Final eval...')
res = evaluate_pu_osls(
    model,
    cfg_prior=cfg,
    unseen_label=unseen_label,
    device=device,
    eval_cfg=eval_cfg,
)
print_results(res)


# Save trained model weights
ckpt_path = 'smoke_model.pt'
torch.save(model.state_dict(), ckpt_path)
print(f'Saved model to {ckpt_path}')


step     1/100 | loss 1.0629 | avg10 1.0629 | splits (76, 49, 89, 75) | seen_counts (3, 1, 2, 2) | x (4, 185, 4)

Eval at step 5...
n_tasks           : 200.0000
outlier_rate_test : 0.1934
outlier_auc       : 0.2446
outlier_ap        : 0.1224
tpr@fpr=0.01      : 0.0000
tpr@fpr=0.05      : 0.0000
tpr@fpr=0.10      : 0.0000
seen_acc          : 0.5079
seen_bal_acc      : 0.5347
overall_acc       : 0.4096
overall_bal_acc   : 0.4010
step    10/100 | loss 1.0591 | avg10 1.0594 | splits (51, 75, 75, 62) | seen_counts (2, 2, 2, 2) | x (4, 139, 3)

Eval at step 10...
n_tasks           : 200.0000
outlier_rate_test : 0.1934
outlier_auc       : 0.2548
outlier_ap        : 0.1239
tpr@fpr=0.01      : 0.0000
tpr@fpr=0.05      : 0.0000
tpr@fpr=0.10      : 0.0000
seen_acc          : 0.5073
seen_bal_acc      : 0.5301
overall_acc       : 0.4091
overall_bal_acc   : 0.3976

Eval at step 15...
n_tasks           : 200.0000
outlier_rate_test : 0.1934
outlier_auc       : 0.2613
outlier_ap        : 0.1249
tpr@fpr